# 01 — Video I/O

Everything in the pipeline starts with reading frames from a video file.  
This notebook covers:
- How OpenCV reads video files
- What a video frame actually is (a NumPy array)
- How to display frames inside a notebook
- How to write frames back to disk

**Connection to the pipeline:** `football_analysis/utils/video_utils.py` → called first in `main.py`

In [ ]:
import sys
sys.path.append('..')  # so we can import football_analysis

import cv2
import numpy as np
import matplotlib.pyplot as plt

## Reading a video

`cv2.VideoCapture` opens a video file (or webcam).  
`.read()` returns one frame at a time as a BGR NumPy array — note **BGR**, not RGB.

In [ ]:
VIDEO_PATH = '../input_videos/08fd33_4.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'Frames : {total_frames}')
print(f'FPS    : {fps}')
print(f'Size   : {width} x {height}')
print(f'Duration: {total_frames / fps:.1f} s')

## What is a frame?

A frame is a 3-D NumPy array: `(height, width, 3)` where the 3 channels are **B, G, R** (not R, G, B).  
Matplotlib expects RGB, so we need to reverse the channel order with `[:, :, ::-1]`.

In [ ]:
ret, frame = cap.read()  # read the first frame
cap.release()

print(f'Array shape : {frame.shape}')   # (H, W, 3)
print(f'dtype       : {frame.dtype}')   # uint8 — values 0-255
print(f'Min / Max   : {frame.min()} / {frame.max()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(frame)                    # wrong colours (BGR treated as RGB)
axes[0].set_title('BGR read directly (wrong colours)')
axes[0].axis('off')

axes[1].imshow(frame[:, :, ::-1])        # channel-reversed → correct
axes[1].set_title('Corrected to RGB')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Reading all frames at once

Our pipeline uses `read_video()` which collects every frame into a Python list.  
This is simple but memory-hungry (750 frames × ~6 MB each ≈ 4.5 GB).

For the **annotation pass** we later switched to streaming (one frame at a time) to avoid OOM — see `_stream_annotated_video` in `main.py`.

In [ ]:
from football_analysis.utils.video_utils import read_video

frames = read_video(VIDEO_PATH)
print(f'Loaded {len(frames)} frames')
print(f'Each frame: {frames[0].shape}')

## Showing a strip of frames across time

In [ ]:
sample_indices = [0, 150, 300, 450, 600, 749]
fig, axes = plt.subplots(1, len(sample_indices), figsize=(18, 3))

for ax, idx in zip(axes, sample_indices):
    ax.imshow(frames[idx][:, :, ::-1])
    ax.set_title(f'Frame {idx}')
    ax.axis('off')

plt.suptitle('Sampled frames across the clip', y=1.02)
plt.tight_layout()
plt.show()

## Writing frames to a video file

`cv2.VideoWriter` writes frames one at a time.  
Key args: fourcc codec (XVID → AVI), fps, and frame size `(width, height)` — note width first, opposite to NumPy's `(H, W)`.

In [ ]:
OUT_PATH = '/tmp/sample_5frames.avi'

h, w = frames[0].shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(OUT_PATH, fourcc, fps, (w, h))

for frame in frames[:5]:   # write only first 5 frames
    out.write(frame)        # frame must stay in BGR

out.release()
print(f'Written to {OUT_PATH}')

## Key takeaways

| Fact | Why it matters |
|---|---|
| Frames are `uint8` BGR arrays | Always convert to RGB before showing with matplotlib |
| `VideoWriter` takes `(width, height)`, NumPy uses `(height, width)` | Easy source of silent bugs |
| Loading all frames at once costs ~4.5 GB for this clip | Motivated the streaming approach in the pipeline |

**Next:** `02_detection.ipynb` — running YOLO on a single frame